# CarePath DARAG - Part 2: Train + Evaluate (GPU)

Runs on **Colab (T4/L4) or a local NVIDIA GPU** - the setup cell auto-detects and
reports your GPU. It restores the Part 1 artifacts (from Drive on Colab, from disk
locally) and runs the full DARAG method:

1. Synthetic in-domain transcripts (few-shot LLM, Sec 4.1 Step 1)
2. Voice-cloning TTS conditioned on in-domain speakers (Sec 4.1 Step 2 / App. D)
3. Synthetic GEC pairs via Gipformer over the cloned audio (Sec 4.1 Step 3)
4. Leakage report - cosine + BLEU vs real (App. C / Table 6)
5. QLoRA fine-tune (full + w/o-RAC / w/o-Aug / only-Synth ablations, Sec 5)
6. Predict + WER and NE-F1 tables (Tables 3 & 4) + acceptance gate

**Checkpoints:** training auto-resumes from the newest checkpoint in the adapters
dir, so a Colab disconnect (or closing a local laptop) continues instead of
restarting. On Colab that dir is on Drive.

In [ ]:
!nvidia-smi
!pip install -q "datasets[audio]" soundfile huggingface_hub sherpa-onnx numpy jiwer \
  transformers accelerate peft trl bitsandbytes sentence-transformers pyvi sacrebleu
# Voice cloning (viXTTS / XTTS-v2). If this wheel is slow/unavailable, the TTS step
# falls back to single-speaker MMS (labeled tts_provider="mms_no_clone").
!pip install -q coqui-tts || pip install -q TTS

## 1. Set up the environment (Colab or local GPU)

In [ ]:
# Set up the repo path + detect the runtime. Works on Colab AND a local machine.
import os, sys, subprocess, zipfile, importlib.util
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

def _find_repo(start: Path):
    for d in [start, *start.parents]:
        if (d / 'pyproject.toml').exists() and (d / 'apps' / 'api' / 'carepath').exists():
            return d
    return None

REPO = _find_repo(Path.cwd())
if REPO is None and IN_COLAB:
    target = Path('/content/carepath')
    zip_path = os.environ.get('CAREPATH_REPO_ZIP', '/content/carepath.zip')
    repo_url = os.environ.get('CAREPATH_REPO_URL')
    if (target / 'pyproject.toml').exists():
        REPO = target
    elif Path(zip_path).exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall('/content/carepath_unzip')
        roots = [p.parent for p in Path('/content/carepath_unzip').rglob('pyproject.toml')]
        src = roots[0] if roots else Path('/content/carepath_unzip')
        target.mkdir(exist_ok=True)
        subprocess.run(f"cp -r '{src}'/* '{target}'/", shell=True, check=True)
        REPO = target
    elif repo_url:
        subprocess.run(['git', 'clone', repo_url, str(target)], check=True)
        REPO = target
    else:
        raise SystemExit('Colab: upload carepath.zip to /content, or set CAREPATH_REPO_ZIP / CAREPATH_REPO_URL.')

if REPO is None:
    raise SystemExit('Could not find the CarePath repo. Open this notebook from inside the cloned repo.')

os.chdir(REPO)
sys.path.insert(0, str(REPO / 'apps' / 'api'))
print(('Colab' if IN_COLAB else 'Local'), '| repo:', REPO)

In [ ]:
# Helper: run a pipeline CLI with PYTHONPATH set, streaming output, raising on failure.
import os, subprocess, sys

def run_step(args, env_extra=None):
    env = dict(os.environ)
    env["PYTHONPATH"] = "apps/api"
    env["PYTHONIOENCODING"] = "utf-8"
    if env_extra:
        env.update(env_extra)
    print(">>>", " ".join(args), flush=True)
    proc = subprocess.run([sys.executable, *args], env=env)
    if proc.returncode != 0:
        raise RuntimeError(f"step failed ({proc.returncode}): {' '.join(args)}")

In [ ]:
from carepath.gec.env import gpu_report, setup_backup
print(gpu_report())
# Drive on Colab (so progress survives a disconnect); None locally (disk persists).
BACKUP = setup_backup(IN_COLAB)
# Adapters + Trainer checkpoints. On Colab they live on Drive so an interrupted
# run resumes after reconnect; locally they persist in the repo.
ADAPTERS = str((BACKUP / 'gec_lora' / 'qwen3') if BACKUP else Path('artifacts/gec_lora/qwen3'))
print('Adapters/checkpoints ->', ADAPTERS)

## 2. Restore Part 1 artifacts (Drive on Colab, disk locally)

In [ ]:
from carepath.gec.env import restore_artifacts
DATASTORE = 'artifacts/retrieval/term_datastore.json'
PAIRS = 'artifacts/gec_pairs/vimedcss_gipformer_pairs.jsonl'
# Colab: copy from Drive. Local: confirm Part 1 already wrote them to disk.
restore_artifacts(BACKUP, [DATASTORE, PAIRS])

## 3. Run size - keep smoke defaults, raise for a real run

In [ ]:
SYNTH_COUNT = 50       # synthetic transcripts (paper nsyn = n)
SYNTH_TTS_LIMIT = 10   # how many to voice-clone + ASR for a smoke run
NSYN_FACTOR = 1.0      # synthetic pairs added to train = factor * |real train|
MAX_STEPS = 60         # raise (e.g. 300+) for a real fine-tune
TTS_PROVIDER = 'xtts'  # 'mms' for the single-speaker fallback
SYNTH_CLEAN = 'artifacts/synthetic/synthetic_clean.jsonl'
TTS_MANIFEST = 'artifacts/synthetic/synthetic_audio_manifest.jsonl'
SYNTH_PAIRS = 'artifacts/gec_pairs/darag_synthetic_pairs.jsonl'
AUGMENTED = 'artifacts/gec_pairs/darag_augmented.jsonl'
print('MAX_STEPS =', MAX_STEPS, '| SYNTH_COUNT =', SYNTH_COUNT, '| training auto-resumes from checkpoints')

## 4. Synthetic in-domain transcripts (paper Sec 4.1 Step 1)

In [ ]:
run_step([
    "scripts/gec/gen_synthetic.py",
    "--pairs", PAIRS,
    "--output", SYNTH_CLEAN,
    "--count", str(SYNTH_COUNT),
    "--load-in-4bit",
])

## 5. Voice-cloning TTS + synthetic GEC pairs (paper Sec 4.1 Steps 2-3)

In [ ]:
run_step([
    "scripts/gec/voice_clone_tts.py",
    "--input", SYNTH_CLEAN,
    "--output", TTS_MANIFEST,
    "--provider", TTS_PROVIDER,
    "--ref-dataset", "tensorxt/ViMedCSS",
    "--ref-count", "20",
    "--limit", str(SYNTH_TTS_LIMIT),
    "--resume",
])
run_step([
    "scripts/gec/make_synth_pairs.py",
    "--input", TTS_MANIFEST,
    "--output", SYNTH_PAIRS,
    "--datastore", DATASTORE,
    "--resume",
])

## 6. Leakage report - synthetic is in-domain but not memorized (paper Table 6)

In [ ]:
run_step([
    "scripts/gec/check_leakage.py",
    "--synthetic", SYNTH_CLEAN,
    "--real", PAIRS,
    "--output", "artifacts/evaluations/leakage.json",
])

## 7. Augment training set + QLoRA fine-tune all variants (paper Sec 5)

Training auto-resumes from any checkpoint in the adapters dir (`--no-resume` to force fresh).

In [ ]:
run_step([
    "scripts/gec/augment.py",
    "--real", PAIRS,
    "--synthetic", SYNTH_PAIRS,
    "--output", AUGMENTED,
    "--nsyn-factor", str(NSYN_FACTOR),
])
# Trains full + wo_rac + wo_aug + only_synth into <ADAPTERS>/<variant>.
# For a fast smoke run, drop --all-variants to train just the full adapter.
run_step([
    "scripts/gec/train.py",
    "--pairs", AUGMENTED,
    "--output-dir", ADAPTERS,
    "--all-variants",
    "--max-steps", str(MAX_STEPS),
])

## 8. LLM/RAG baseline + trained predictions

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "offline")
EVAL_SPLIT = AUGMENTED  # score on the frozen val/test/hard rows inside the file
run_step([
    "scripts/gec/llm_rag_baseline.py",
    "--input", PAIRS,
    "--output", "artifacts/evaluations/llm_rag.jsonl",
])
# Run the full DARAG adapter over the LLM/RAG output so one file has every column.
run_step([
    "scripts/gec/predict.py",
    "--pairs", "artifacts/evaluations/llm_rag.jsonl",
    "--adapter-dir", f"{ADAPTERS}/full",
    "--output", "artifacts/evaluations/darag_all_preds.jsonl",
    "--column", "gec_pred",
])

## 9. WER + NE-F1 tables (paper Tables 3 & 4) + acceptance gate

In [ ]:
run_step([
    "scripts/gec/evaluate.py",
    "--input", "artifacts/evaluations/darag_all_preds.jsonl",
    "--prediction-columns", "raw_asr", "corrected_text", "gec_pred",
    "--wer-output", "artifacts/evaluations/darag_wer.json",
    "--ne-f1-output", "artifacts/evaluations/darag_ne_f1.json",
])
# Gate: trained adapter must match-or-beat raw + LLM/RAG on val+hard (non-zero on REJECT).
run_step(["scripts/gec/gate.py", "--report", "artifacts/evaluations/darag_wer.json"])

## 10. Save adapters + metrics (Drive on Colab, disk locally)

In [ ]:
from carepath.gec.env import save_artifacts
save_artifacts(BACKUP, [
    'artifacts/evaluations/darag_wer.json',
    'artifacts/evaluations/darag_ne_f1.json',
    'artifacts/evaluations/leakage.json',
])
# On Colab the adapters already live on Drive (ADAPTERS points there);
# locally they persist in the repo's artifacts/gec_lora.
print('Adapters at:', ADAPTERS)
print('Part 2 done.')

## 11. Bundle the adapter for the student (Google Drive)

In [ ]:
from carepath.gec.env import bundle_adapter
# Package ONLY the final adapter (not the big checkpoint-* dirs) into a small,
# student-ready folder + zip. On Colab this lands on Drive next to the artifacts.
share_root = (BACKUP / 'share') if BACKUP else Path('artifacts/share')
folder, zip_path = bundle_adapter(f'{ADAPTERS}/full', share_root, name='carepath-gec-qwen3-full')
print('Shareable folder:', folder)
print('Zip:', zip_path)
if BACKUP:
    print('\nShare with the student (one-time, in the Google Drive UI):')
    print('  1. Open Google Drive -> MyDrive/carepath_artifacts/share')
    print('  2. Right-click carepath-gec-qwen3-full.zip -> Share -> Anyone with the link (Viewer)')
    print('  3. Send the link; they paste it into the CarePath_use_adapter notebook.')
else:
    print('\nShare the folder/zip above; the student points ADAPTER_DIR at it.')